# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

데이터 → 전처리 → 시각화 → EDA → Feature → 분류 → 평가 → 모델 선택 → 저장 → Streamlit

> 코드는 AI의 도움을 받아 최소한으로 작성하지만, 분석을 단계별로 진행하고 실제 결과를 보고 다음 행동을 결정하는 사람은 학생입니다.

각 STEP은 **실행 계획 → 코드 실행 → 결과 요약 → 관찰/해석/가설/한계** 순서로 진행합니다. AI가 실제 실행 결과를 만들어 냈다고 가정하지 않습니다.


## STEP 00. 전체 분석 지도 먼저 보기
STEP 00~17 흐름을 먼저 확인합니다. 개인 판단 지점은 STEP 03, 05, 06, 07, 08, 09, 10, 12, 13, 14, 15, 16, 17이며 **최소 5곳 이상에서 자신의 선택과 이유를 기록**합니다.


## STEP 01. 실행 환경 확인


In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, sklearn
from IPython.display import display
print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("pandas / numpy / sklearn:", pd.__version__, np.__version__, sklearn.__version__)


### STEP 01 기록 — 학생 작성
- 관찰:
- 해석:
- 추가 확인:
- 한계:


## STEP 02. Titanic 데이터 준비와 로딩
먼저 저장소 루트 터미널에서 `python scripts/prepare_titanic_data.py`를 실행합니다. CSV를 임의 생성하지 않습니다.


In [ ]:
def get_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "data").is_dir():
            return path
    raise FileNotFoundError("data 폴더가 있는 프로젝트 루트를 찾을 수 없습니다.")

project_root = get_project_root()
data_path = project_root / "data" / "titanic" / "train.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"{data_path}\n저장소 루트에서 python scripts/prepare_titanic_data.py 를 먼저 실행하세요."
    )
df = pd.read_csv(data_path)
print("shape:", df.shape)
display(df.head())


### STEP 02 기록 — 학생 작성
- 관찰:
- 해석:
- 추가 확인:
- 한계:


## STEP 03. 데이터 구조와 품질의 첫인상 확인


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_rate": (df.isna().mean()*100).round(2),
    "nunique": df.nunique(dropna=False),
})
display(quality)
display(df.describe(include="number").T)
for c in ["Survived","Pclass","Sex","Embarked"]:
    print("\n", c)
    display(df[c].value_counts(dropna=False).rename("count").to_frame())


### STEP 03 개인 판단
추가 품질 확인 1가지를 직접 선택합니다. 예: 중복 행, Fare=0, 이름/티켓 패턴.
- 내가 선택한 확인:
- 이유:
- 관찰:
- 다음 행동:


## STEP 04. 분석 문제와 Target 정의


In [ ]:
TARGET = "Survived"
print(df[TARGET].value_counts().sort_index())
print({"task":"binary_classification","target":TARGET,"positive_class":1})


### STEP 04 기록 — 학생 작성
- 관찰:
- 해석:
- 누수 위험 후보:
- 한계:


## STEP 05. 결측치 처리
`df_work`는 STEP 05~10 탐색/EDA용입니다. 여기서 계산한 중앙값/최빈값은 **최종 모델 평가 전처리가 아닙니다**.


In [ ]:
df_work = df.copy()
age_median_for_eda = df_work["Age"].median()
embarked_mode_for_eda = df_work["Embarked"].mode(dropna=True).iloc[0]
df_work["Age"] = df_work["Age"].fillna(age_median_for_eda)
df_work["Embarked"] = df_work["Embarked"].fillna(embarked_mode_for_eda)
print("EDA Age median:", age_median_for_eda)
print("EDA Embarked mode:", embarked_mode_for_eda)
display(df_work[["Age","Embarked","Cabin"]].isna().sum().to_frame("missing"))


### STEP 05 개인 판단
- 결측치 전략을 유지/변경한 이유:
- 탐색용 처리와 최종 모델용 처리의 차이:
- 추가 확인:


## STEP 06. 불필요한 컬럼 검토


In [ ]:
column_policy = pd.DataFrame({
    "column": ["PassengerId","Name","Ticket","Cabin"],
    "default_policy": ["exclude","derive_or_defer","derive_or_defer","derive_or_defer"],
})
display(column_policy)


### STEP 06 개인 판단
- 유지/제외/파생/보류 정책:
- 이유:
- 나중에 다시 볼 컬럼:


## STEP 07. 범주형 데이터 변환 원리
`df_encoded`는 교육용 임시 branch입니다. **최종 모델 입력으로 사용하지 않습니다.**


In [ ]:
df_encoded = df_work.copy()
df_encoded = pd.get_dummies(df_encoded, columns=["Sex","Embarked"], dtype=int)
display(df_encoded.filter(regex="^(Sex_|Embarked_)").head())


### STEP 07 개인 판단
- one-hot encoding이 필요한 이유:
- 최종 모델에서는 왜 Pipeline 안에서 처리해야 하는가:


## STEP 08. 시각화와 기본 패턴 확인


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots()
sns.barplot(data=df_work, x="Sex", y="Survived", ax=ax)
ax.set_title("Survival rate by Sex")
plt.show()

fig, ax = plt.subplots()
sns.barplot(data=df_work, x="Pclass", y="Survived", ax=ax)
ax.set_title("Survival rate by Pclass")
plt.show()


In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df_work, x="Survived", y="Fare", ax=ax)
ax.set_title("Fare distribution by survival")
ax.set_xlabel("Survived")
ax.set_ylabel("Fare")
plt.show()


### boxplot 읽는 법
- 상자의 아래/위 선: Q1(25%) / Q3(75%), 상자 안 가로선: median(중앙값)
- IQR = Q3 - Q1, whisker(수염): 대략적인 값의 범위
- whisker 밖의 점: outlier candidate(이상치 후보) — **자동으로 삭제할 대상이 아닙니다.** 실제로 운임이 높았던 승객일 수 있습니다.


### STEP 08 개인 판단
- 추가 시각화 1개:
- 관찰:
- 해석:
- 추가 확인:


## STEP 09. 기초 통계와 EDA


In [ ]:
display(df_work.groupby("Sex").agg(rows=("Survived","size"), survival_rate=("Survived","mean")))
display(df_work.groupby("Pclass").agg(rows=("Survived","size"), survival_rate=("Survived","mean")))
display(df_work.groupby(["Sex","Pclass"]).agg(rows=("Survived","size"), survival_rate=("Survived","mean")))


### Welch t-test — 실행 전 개념 확인
- 귀무가설(H0): 생존 그룹과 비생존 그룹의 Fare 평균은 같다.
- p-value: 귀무가설이 참이라고 가정할 때, 지금 관찰한 정도(또는 더 극단적인) 차이가 우연히 나타날 확률.
- 관례적으로 p-value < 0.05면 "통계적으로 유의하다"고 표현하지만, 0.05는 **절대적 기준이 아니라 관례적 기준**입니다.
- p-value가 작다고 해서 Fare가 생존의 **원인**이라는 뜻은 아닙니다(인과관계가 증명되는 것이 아닙니다).
- Fare는 분포가 한쪽으로 치우쳐 있고 매우 큰 값(이상치)도 있으므로, t-test 숫자만 보지 말고 위 boxplot·평균/중앙값 같은 요약 통계와 함께 해석합니다.


In [ ]:
from scipy.stats import ttest_ind

survived_fare = df_work.loc[df_work["Survived"] == 1, "Fare"]
not_survived_fare = df_work.loc[df_work["Survived"] == 0, "Fare"]
t_stat, p_value = ttest_ind(survived_fare, not_survived_fare, equal_var=False)
print("survived Fare mean:", round(survived_fare.mean(), 2), " n =", survived_fare.shape[0])
print("not survived Fare mean:", round(not_survived_fare.mean(), 2), " n =", not_survived_fare.shape[0])
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.3e}")


### STEP 09 개인 판단
- 추가 EDA 질문:
- 관찰:
- 해석:
- 가설:
- 한계:


## STEP 10. Feature 설계
EDA 아이디어를 바로 모델에 넣지 말고 **결정적 row-wise 규칙**과 **데이터에서 학습해야 하는 규칙**을 구분합니다.


In [ ]:
feature_demo = df_work.copy()
feature_demo["FamilySize"] = feature_demo["SibSp"] + feature_demo["Parch"] + 1
feature_demo["IsAlone"] = (feature_demo["FamilySize"] == 1).astype(int)
display(feature_demo[["SibSp","Parch","FamilySize","IsAlone","Survived"]].head())


### STEP 10 → STEP 11 Feature Contract — 학생 작성
기본 실행 Contract는 `FamilySize`, `IsAlone`입니다. 바꾸려면 재현 가능한 고정 규칙인지 먼저 확인합니다.
- 사용할 결정적 Feature:
- 보류한 Feature:
- 학습형/전역 규칙 여부:
- 선택 이유:

> 교육적 한계: EDA와 Feature 아이디어를 전체 데이터에서 먼저 본 뒤 split하므로 이후 holdout은 완전히 손대지 않은 test라고 볼 수 없습니다. 반복 선택은 train 내부 CV로 보강합니다.


## STEP 11. 학습/테스트 데이터 준비
탐색용 `df_work`/`df_encoded`를 버리고 **원본 `df`에서 `model_source`를 다시 만듭니다.** learned preprocessing보다 먼저 split합니다.


In [ ]:
import sys
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from titanic_app.features import (
    RAW_INPUT_COLUMNS, MODEL_FEATURE_COLUMNS,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    build_model_features,
)
from sklearn.model_selection import train_test_split

model_source = build_model_features(df.copy())
X = model_source[MODEL_FEATURE_COLUMNS].copy()
y = model_source["Survived"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("train:", X_train.shape, y_train.shape)
print("test :", X_test.shape, y_test.shape)
display(y_train.value_counts(normalize=True).sort_index().rename("train_ratio").to_frame())
display(y_test.value_counts(normalize=True).sort_index().rename("test_ratio").to_frame())


### STEP 11 기록 — 학생 작성
- split 전에 학습형 전처리를 하지 않았는가:
- train/test 비율:
- Target 비율 확인:
- 한계:


## STEP 12. Baseline 분류 모델 학습
Baseline은 `ColumnTransformer + Pipeline + LogisticRegression`입니다. imputer/OHE/scaler는 `X_train`에만 fit됩니다.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
baseline_pipeline.fit(X_train, y_train)
print("baseline fitted")


In [ ]:
def positive_class_probability(fitted_pipeline, X_input, positive_class=1):
    classes = fitted_pipeline.named_steps["model"].classes_
    positions = np.where(classes == positive_class)[0]
    if len(positions) != 1:
        raise ValueError(f"positive class {positive_class} not found in {classes}")
    return fitted_pipeline.predict_proba(X_input)[:, positions[0]]

baseline_pred = baseline_pipeline.predict(X_test)
baseline_prob = positive_class_probability(baseline_pipeline, X_test)
print("classes:", baseline_pipeline.named_steps["model"].classes_)


### STEP 12 개인 판단
- Baseline을 다음 단계로 가져갈 이유:
- 가장 먼저 확인할 성능/오류 지표:


## STEP 13. 모델 성능 평가와 오류 분석


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

baseline_accuracy = accuracy_score(y_test, baseline_pred)
print("Baseline accuracy:", round(baseline_accuracy, 4))
print("\nConfusion matrix\n", confusion_matrix(y_test, baseline_pred))
print("\nClassification report\n", classification_report(y_test, baseline_pred, digits=4))

error_analysis = X_test.copy()
error_analysis["actual"] = y_test.to_numpy()
error_analysis["predicted"] = baseline_pred
error_analysis["survival_probability"] = baseline_prob
error_analysis["correct"] = error_analysis["actual"] == error_analysis["predicted"]
display(error_analysis.loc[~error_analysis["correct"]].head(15))


### STEP 13 개인 판단
- 어떤 종류의 오류가 중요한가:
- 대표 오분류 패턴:
- 추가 확인:
- 이 결과만으로 말할 수 없는 것:


## STEP 14. 추가 모델/알고리즘 선택
Titanic은 Target이 있는 분류 문제이므로 **unsupervised를 억지로 넣지 않습니다.** AI에게 후보 3개와 장단점을 요청하고 학생이 선택합니다. 기본 실행 예시는 RandomForest이며, 선택에 따라 estimator만 교체할 수 있습니다.


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

additional_estimator = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
)
additional_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", additional_estimator),
])
additional_pipeline.fit(X_train, y_train)
additional_pred = additional_pipeline.predict(X_test)
additional_accuracy = accuracy_score(y_test, additional_pred)
print("Additional model:", additional_estimator.__class__.__name__)
print("Accuracy:", round(additional_accuracy, 4))


### STEP 14 개인 판단
- AI가 제안한 후보:
- 내가 선택한 모델:
- 선택 이유:
- 복잡도/해석 가능성 고려:


## STEP 15. 모델 비교와 최종 모델 선정
같은 split/Feature contract를 유지합니다. 반복 선택의 근거는 **X_train/y_train 내부 5-Fold CV**로 보강합니다. holdout test를 계속 튜닝에 사용하지 않습니다.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_cv = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
additional_cv = cross_val_score(additional_pipeline, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

comparison = pd.DataFrame({
    "model": ["LogisticRegression", additional_estimator.__class__.__name__],
    "cv_mean_accuracy": [baseline_cv.mean(), additional_cv.mean()],
    "cv_std": [baseline_cv.std(), additional_cv.std()],
    "holdout_accuracy": [baseline_accuracy, additional_accuracy],
})
display(comparison.round(4))


In [ ]:
# 실제 비교 결과와 해석을 확인한 뒤 학생이 변경합니다.
FINAL_MODEL_CHOICE = "baseline"  # "baseline" 또는 "additional"

if FINAL_MODEL_CHOICE == "baseline":
    final_pipeline = baseline_pipeline
elif FINAL_MODEL_CHOICE == "additional":
    final_pipeline = additional_pipeline
else:
    raise ValueError("FINAL_MODEL_CHOICE must be 'baseline' or 'additional'.")

print("Final estimator:", final_pipeline.named_steps["model"].__class__.__name__)


### Model Input Contract
- raw input: `Pclass, Sex, Age, SibSp, Parch, Fare, Embarked`
- fixed derived: `FamilySize, IsAlone`
- learned preprocessing: Pipeline 내부에서 train data로 fit
- target: `Survived`, positive class = 1

### STEP 15 개인 판단
- 최종 모델:
- CV 근거:
- holdout 근거:
- 정확도 외 고려사항:
- 선택 이유:


## STEP 16. 최종 Pipeline 저장과 새로운 승객 예측


In [ ]:
import json, joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
pipeline_path = models_dir / "titanic_final_pipeline.joblib"
contract_path = models_dir / "titanic_model_contract.json"

joblib.dump(final_pipeline, pipeline_path)
model_contract = {
    "target": "Survived",
    "positive_class": 1,
    "raw_input_columns": RAW_INPUT_COLUMNS,
    "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "derived_features": {
        "FamilySize": "SibSp + Parch + 1",
        "IsAlone": "1 if FamilySize == 1 else 0",
    },
    "final_estimator": final_pipeline.named_steps["model"].__class__.__name__,
}
contract_path.write_text(json.dumps(model_contract, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", pipeline_path)
print("saved:", contract_path)


In [ ]:
loaded_pipeline = joblib.load(pipeline_path)
loaded_contract = json.loads(contract_path.read_text(encoding="utf-8"))
if not np.array_equal(loaded_pipeline.predict(X_test), final_pipeline.predict(X_test)):
    raise RuntimeError("저장 전/후 예측이 다릅니다.")
print("Reload prediction check: PASS")
display(loaded_contract)


In [ ]:
new_passenger_raw = pd.DataFrame([{
    "Pclass": 3, "Sex": "male", "Age": 30.0,
    "SibSp": 0, "Parch": 0, "Fare": 10.0, "Embarked": "S",
}])
new_X = build_model_features(new_passenger_raw)[MODEL_FEATURE_COLUMNS]
new_prediction = int(loaded_pipeline.predict(new_X)[0])
new_probability = float(positive_class_probability(loaded_pipeline, new_X, 1)[0])
print("Predicted class:", new_prediction)
print("Survival probability:", round(new_probability, 4))


### STEP 16 개인 판단
- 저장/재로드 확인:
- 새 승객 입력이 Contract와 맞는가:
- 예측을 실제 인과관계처럼 해석하면 안 되는 이유:


## STEP 17. Streamlit 예측 앱 구현
`src/titanic_app/app.py`는 저장된 Pipeline과 `features.py`를 사용합니다. 앱에서 median/mode 계산, `pd.get_dummies()`, scaler fit을 다시 하지 않습니다.

```powershell
streamlit run src/titanic_app/app.py
```


In [ ]:
app_path = project_root / "src" / "titanic_app" / "app.py"
features_path = project_root / "src" / "titanic_app" / "features.py"
print("app:", app_path.is_file(), app_path)
print("features:", features_path.is_file(), features_path)
print("model:", pipeline_path.is_file(), pipeline_path)
print("contract:", contract_path.is_file(), contract_path)


### STEP 17 개인 판단
- UI에서 추가/제거할 입력:
- 예측 결과와 함께 보여줄 설명:
- 사용자에게 반드시 알려야 할 한계:

## 최종 회고
- 가장 중요한 관찰:
- 내가 직접 내린 판단 5개 이상:
- 모델 선택 이유:
- 분석에서 가장 큰 한계:
- 다음에 다시 한다면 바꿀 점:

## 다음 단계: Data Analysis Agent
향후에는 `공개 데이터/API/크롤링 수집 → 품질 검증 → 분석 질문 생성 → 단계별 실행 → 결과 기반 다음 분석 선택 → 보고서 자동화`로 확장할 수 있습니다. 핵심은 분석 흐름을 AI가 임의로 끝까지 대신하는 것이 아니라, **각 단계의 실제 결과와 검증 기준을 다음 행동의 입력으로 사용하는 것**입니다.
